<a href="https://colab.research.google.com/github/HannahShaw21/Public-Repo-4-Hannah/blob/main/HW4_Hannah_Shaw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Road the California housing data into your ipynb by using the following code:

In [31]:
from sklearn.datasets import fetch_california_housing
ds = fetch_california_housing()
X, y = ds.data, ds.target

2. What is the dimension of X? What is the size of the data (n)? What values does X and y
have? Do print(california_housing.DESCR . What are the meanings of each dimension (variable)
in X?

In [32]:
X.shape

(20640, 8)

In [33]:
X.size

165120

In [34]:
X

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]])

In [35]:
y

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894])

In [36]:
print(ds.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

3. By using “from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV”, apply Ridge,
Lass, Elastic with cross-validation.

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import numpy as np
import pandas as pd

#Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Define helps
def evaluate(model, name):
  y_pred_tr = model.predict(X_train)
  y_pred_te = model.predict(X_test)
  rmse_tr = np.sqrt(mean_squared_error(y_train, y_pred_tr))
  rmse_te = np.sqrt(mean_squared_error(y_test, y_pred_te))
  mae_te = mean_absolute_error(y_test, y_pred_te)
  r2_te = r2_score(y_test, y_pred_te)
  print(f"[{name}]  RMSE(train)={rmse_tr:.3f}  RMSE(test)={rmse_te:.3f}  "
        f"MAE(test)={mae_te:.3f}  R2(test)={r2_te:.3f}")
  return {"name": name, "rmse_train": rmse_tr, "rmse_test": rmse_te, "mae_test": mae_te, "r2_test": r2_te}

results = []

#Ridge with cv
ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5))
])
ridge.fit(X_train, y_train)
results.append(evaluate(ridge, "Ridge"))
best_alpha_ridge = ridge.named_steps["model"].alpha_
print(f"-> Ridge best alpha = {best_alpha_ridge:.5f}")

#Lasso with cv
lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(alphas=np.logspace(-3, 3, 100), max_iter=10000, random_state=42, cv=5))
])
lasso.fit(X_train, y_train)
results.append(evaluate(lasso, "Lasso"))
best_alpha_lasso = lasso.named_steps["model"].alpha_
print(f"-> Lasso best alpha = {best_alpha_lasso:.5f}")

#Elastic net with cv
enet = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        l1_ratio=[0.2, 0.5, 0.8, 0.95, 1.0], #1.0: close to Lasso
        alphas=np.logspace(-3, 3, 50),
        max_iter=10000,
        random_state=42, cv=5))
])
enet.fit(X_train, y_train)
results.append(evaluate(enet, "Elastic Net"))
best_alpha_enet = enet.named_steps["model"].alpha_
best_l1ratio_enet = enet.named_steps["model"].l1_ratio_
print(f"-> Lasso best alpha = {best_alpha_enet:.5f}, "
      f"best l1_ratio = {best_l1ratio_enet}")

[Ridge]  RMSE(train)=0.720  RMSE(test)=0.746  MAE(test)=0.533  R2(test)=0.576
-> Ridge best alpha = 0.00100
[Lasso]  RMSE(train)=0.720  RMSE(test)=0.745  MAE(test)=0.533  R2(test)=0.577
-> Lasso best alpha = 0.00100
[Elastic Net]  RMSE(train)=0.720  RMSE(test)=0.745  MAE(test)=0.533  R2(test)=0.577
-> Lasso best alpha = 0.00100, best l1_ratio = 1.0


4. By using “from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error”
compare the results and interpret.

In [38]:
results_df = pd.DataFrame(results)
print("\nSummary:")
print(results_df)


Summary:
          name  rmse_train  rmse_test  mae_test   r2_test
0        Ridge    0.719676   0.745581  0.533200  0.575788
1        Lasso    0.719715   0.744642  0.533145  0.576856
2  Elastic Net    0.719715   0.744642  0.533145  0.576856


When we compare the results of Ridge, Lasso, and Elastic regression with cross-validation, we see that the results for all three methods are very similar to each other in terms of how well they suit the training data, the test data, the MAE test, and the R-squared test.

5. Road the MNIST data into your ipynb by using the following code:

In [39]:
from sklearn.datasets import fetch_openml
import numpy as np

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist['data'].astype('float32') / 255.00 #shape: (70000, 784)
y = mnist['target'].astype('int64')

6. What is the dimension of X? What is the size of the data (n)? What values does X and y
have? What are the meanings of each dimension (variable) in X? (See, https://huggingface.co/
datasets/ylecun/mnist )

In [40]:
X.shape

(70000, 784)

In [41]:
X.size

54880000

In [42]:
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [43]:
y

array([5, 0, 4, ..., 4, 5, 6])

In [44]:
print(mnist.DESCR)

**Author**: Yann LeCun, Corinna Cortes, Christopher J.C. Burges  
**Source**: [MNIST Website](http://yann.lecun.com/exdb/mnist/) - Date unknown  
**Please cite**:  

The MNIST database of handwritten digits with 784 features, raw data available at: http://yann.lecun.com/exdb/mnist/. It can be split in a training set of the first 60,000 examples, and a test set of 10,000 examples  

It is a subset of a larger set available from NIST. The digits have been size-normalized and centered in a fixed-size image. It is a good database for people who want to try learning techniques and pattern recognition methods on real-world data while spending minimal efforts on preprocessing and formatting. The original black and white (bilevel) images from NIST were size normalized to fit in a 20x20 pixel box while preserving their aspect ratio. The resulting images contain grey levels as a result of the anti-aliasing technique used by the normalization algorithm. the images were centered in a 28x28 image b

7. By using “from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV”, apply Ridge,
Lass, Elastic with cross-validation.

In [47]:
#Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

results = []

#Ridge with cv
ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5))
])
ridge.fit(X_train, y_train)
results.append(evaluate(ridge, "Ridge"))
best_alpha_ridge = ridge.named_steps["model"].alpha_
print(f"-> Ridge best alpha = {best_alpha_ridge:.5f}")

#Lasso with cv
lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(alphas=np.logspace(-3, 3, 100), max_iter=10000, random_state=42, cv=5))
])
lasso.fit(X_train, y_train)
results.append(evaluate(lasso, "Lasso"))
best_alpha_lasso = lasso.named_steps["model"].alpha_
print(f"-> Lasso best alpha = {best_alpha_lasso:.5f}")

#Elastic net with cv
enet = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        l1_ratio=[0.2, 0.5, 0.8, 0.95, 1.0], #1.0: close to Lasso
        alphas=np.logspace(-3, 3, 50),
        max_iter=10000,
        random_state=42, cv=5))
])
enet.fit(X_train, y_train)
results.append(evaluate(enet, "Elastic Net"))
best_alpha_enet = enet.named_steps["model"].alpha_
best_l1ratio_enet = enet.named_steps["model"].l1_ratio_
print(f"-> Lasso best alpha = {best_alpha_enet:.5f}, "
      f"best l1_ratio = {best_l1ratio_enet}")

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.31867e-10): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.86263e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.57529e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=6.2029e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.10136e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python

[Ridge]  RMSE(train)=1.788  RMSE(test)=1.815  MAE(test)=1.406  R2(test)=0.606
-> Ridge best alpha = 1000.00000
[Lasso]  RMSE(train)=1.792  RMSE(test)=1.795  MAE(test)=1.405  R2(test)=0.615
-> Lasso best alpha = 0.00404
[Elastic Net]  RMSE(train)=1.793  RMSE(test)=1.794  MAE(test)=1.407  R2(test)=0.615
-> Lasso best alpha = 0.02223, best l1_ratio = 0.2


8. By using “from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error”
compare the results and interpret.

In [48]:
results_df = pd.DataFrame(results)
print("\nSummary:")
print(results_df)


Summary:
          name  rmse_train  rmse_test  mae_test   r2_test
0        Ridge    1.787531   1.815138  1.406479  0.606416
1        Lasso    1.791744   1.794802  1.404906  0.615185
2  Elastic Net    1.792650   1.794192  1.406686  0.615447


9. Did you get any good regression result for MNIST? Discuss the performance of regression
method on MNIST. Is it proper to apply linear regression method to MNIST or not? If yes, why?
If not, why?

I did not get a good regression result for MNIST, though the results for Ridge, Lasso, and Elastic net linear regression were similiar for all three methods. The performance of the regression methods on MNIST was very poor, as it took over 20 minutes for the code cell in #7 to finish running.

I do not think that applying linear regression method to MNIST would be proper, as MNIST is not a typical dataset of data collected over time/an area, but rather a collection of handwritten numbers translated into numerical arrays, which is very different from the kind of data that linear regression was meant to analyze and would not have an clear linear "pattern" for linear regression models to follow.